In [1]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q numpy
!pip install -q seqeval
!pip install -q "datasets<4.0.0"

In [2]:
from datasets import load_dataset

In [3]:
ds = load_dataset("Goader/ner-uk-2.0")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['document_id', 'tokens', 'ner_tags', 'source'],
        num_rows: 10980
    })
    validation: Dataset({
        features: ['document_id', 'tokens', 'ner_tags', 'source'],
        num_rows: 1206
    })
    test: Dataset({
        features: ['document_id', 'tokens', 'ner_tags', 'source'],
        num_rows: 5593
    })
})


In [ ]:
# Try to get label names directly — may or may not work depending on
# whether this dataset kept its ClassLabel metadata (tner/ontonotes5 lost
# it through parquet auto-conversion, this one might not have)
try:
    label_list = ds["train"].features["ner_tags"].feature.names
    print(label_list)
except AttributeError:
    print("No embedded label names — will need to find them separately.")

print(ds["train"][0])

In [4]:

label_list = [
   'O', 'B-ORG', 'I-ORG', 'B-PERS', 'I-PERS', 'B-LOC', 'I-LOC', 'B-MON', 'I-MON', 'B-PCT', 'I-PCT', 'B-DATE',
    'I-DATE', 'B-TIME', 'I-TIME', 'B-PERIOD', 'I-PERIOD', 'B-JOB', 'I-JOB', 'B-DOC', 'I-DOC', 'B-QUANT', 
    'I-QUANT', 'B-ART', 'I-ART', 'B-MISC', 'I-MISC'
]

In [5]:
new_label_list = ["O", "B-LOC", "I-LOC", "B-DATE", "I-DATE"]
new_label2id = {l: i for i, l in enumerate(new_label_list)}
new_id2label = {i: l for i, l in enumerate(new_label_list)}

def remap(old_name: str) -> str:
    if old_name == "B-LOC":
        return "B-LOC"
    if old_name == "I-LOC":
        return "I-LOC"
    if old_name in ("B-DATE", "B-PERIOD"):
        return "B-DATE"
    if old_name in ("I-DATE", "I-PERIOD"):
        return "I-DATE"
    return "O"

old_id_to_new_id = {old_id: new_label2id[remap(name)] for old_id, name in enumerate(label_list)}

In [ ]:
for i in range(200):
    ex = ds["train"][i]
    tags = [label_list[t] for t in ex["ner_tags"]]
    if any(t in ("B-DATE", "I-DATE") for t in tags):
        print("Приклад №", i)
        for tok, t in zip(ex["tokens"], tags):
            print(tok, "->", t)
        break

In [6]:
def remap_tags(example: dict) -> dict:
    example["ner_tags"] = [old_id_to_new_id[t] for t in example["ner_tags"]]
    return example

ds_remapped = ds.map(remap_tags)

ex = ds_remapped["train"][28]
for tok, tag_id in zip(ex["tokens"], ex["ner_tags"]):
    print(tok, "->", new_label_list[tag_id])

Стефанові -> O
іноді -> O
здавалося -> O
, -> O
що -> O
на -> O
вулиці -> O
— -> O
ніяк -> O
не -> O
2013 -> B-DATE
, -> O
а -> O
1913 -> B-DATE
, -> O
і -> O
він -> O
— -> O
не -> O
студент -> O
із -> O
Відня -> B-LOC
, -> O
а -> O
один -> O
із -> O
друзів -> O
Штукенгайзена -> O
, -> O
що -> O
народився -> O
разом -> O
із -> O
ним -> O
у -> O
Дуклі -> B-LOC
, -> O
але -> O
не -> O
потрапив -> O
на -> O
полотно -> O
, -> O
де -> O
Себастьян -> O
грає -> O
в -> O
бабу-куцю -> O
із -> O
сільськими -> O
дівчатками -> O
, -> O
випадково -> O
. -> O


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_and_align_labels(example: dict) -> dict:
    tokenized = tokenizer(example["tokens"], is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    labels, prev_word_id = [], None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != prev_word_id:
            labels.append(example["ner_tags"][word_id])
        else:
            labels.append(-100)
        prev_word_id = word_id
    tokenized["labels"] = labels
    return tokenized

ds_final = ds_remapped.map(tokenize_and_align_labels)

train_ds_uk = ds_final["train"]
val_ds_uk = ds_final["validation"]
test_ds_uk = ds_final["test"]

print(train_ds_uk[28])

Map:   0%|          | 0/1206 [00:00<?, ? examples/s]

{'document_id': '2936652097f4', 'tokens': ['Стефанові', 'іноді', 'здавалося', ',', 'що', 'на', 'вулиці', '—', 'ніяк', 'не', '2013', ',', 'а', '1913', ',', 'і', 'він', '—', 'не', 'студент', 'із', 'Відня', ',', 'а', 'один', 'із', 'друзів', 'Штукенгайзена', ',', 'що', 'народився', 'разом', 'із', 'ним', 'у', 'Дуклі', ',', 'але', 'не', 'потрапив', 'на', 'полотно', ',', 'де', 'Себастьян', 'грає', 'в', 'бабу-куцю', 'із', 'сільськими', 'дівчатками', ',', 'випадково', '.'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'source': 'bruk', 'input_ids': [0, 42650, 12800, 204831, 210, 28476, 29419, 6, 4, 899, 29, 102777, 292, 56527, 77, 1210, 6, 4, 252, 70351, 6, 4, 189, 7315, 292, 77, 21302, 4459, 15942, 1951, 6, 4, 252, 5641, 4459, 210626, 3681, 24710, 812, 27525, 6309, 212, 6, 4, 899, 220687, 44747, 4459, 15844, 84, 16372, 698, 1536, 6, 4, 3573, 77, 129, 6574, 151322, 

In [8]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, accuracy_score


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [new_label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [new_label_list[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
# model_uk = AutoModelForTokenClassification.from_pretrained(
#     "xlm-roberta-base",
#     num_labels=len(new_label_list),
#     id2label=new_id2label,
#     label2id=new_label2id,
# )
# 
# training_args = TrainingArguments(
#     output_dir="./results_uk",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     logging_strategy="epoch",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
# )
# 
# trainer_uk = Trainer(
#     model=model_uk,
#     args=training_args,
#     train_dataset=train_ds_uk,
#     eval_dataset=val_ds_uk,
#     processing_class=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
# )
# 
# trainer_uk.train()

In [12]:
model_uk_v2 = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(new_label_list),
    id2label=new_id2label,
    label2id=new_label2id,
)

training_args_v2 = TrainingArguments(
    output_dir="./results_uk_v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,   # NEW: keep only the single best checkpoint on disk, delete the rest automatically
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_uk_v2 = Trainer(
    model=model_uk_v2,
    args=training_args_v2,
    train_dataset=train_ds_uk,
    eval_dataset=val_ds_uk,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_uk_v2.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vect

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.207223,0.066844,0.566820,0.768750,0.652520,0.987261
2,0.045735,0.051738,0.723164,0.800000,0.759644,0.990560
3,0.029911,0.050309,0.735955,0.818750,0.775148,0.991728
4,0.020303,0.069697,0.732759,0.796875,0.763473,0.991271
5,0.014782,0.065197,0.764881,0.803125,0.783537,0.992032
6,0.010459,0.065095,0.768546,0.809375,0.788432,0.992184


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2064, training_loss=0.05473534462525863, metrics={'train_runtime': 1231.3807, 'train_samples_per_second': 53.501, 'train_steps_per_second': 1.676, 'total_flos': 3355165852140960.0, 'train_loss': 0.05473534462525863, 'epoch': 6.0})

In [10]:
!df -h /kaggle/working
!du -sh ~/.cache/huggingface 2>/dev/null
!du -sh /kaggle/working/* 2>/dev/null

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   20G     0 100% /kaggle/working
1.1G	/root/.cache/huggingface
9.4G	/kaggle/working/results_uk
11G	/kaggle/working/results_uk_v2


In [11]:
!rm -rf /kaggle/working/results_uk
!rm -rf /kaggle/working/results_uk_v2
!df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   76K   20G   1% /kaggle/working


In [13]:
model_uk_v3 = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(new_label_list),
    id2label=new_id2label,
    label2id=new_label2id,
)

training_args_v3 = TrainingArguments(
    output_dir="./results_uk_v3",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_uk_v3 = Trainer(
    model=model_uk_v3,
    args=training_args_v3,
    train_dataset=train_ds_uk,
    eval_dataset=val_ds_uk,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_uk_v3.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vect

Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 734.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 290.81 MiB is free. Including non-PyTorch memory, this process has 14.28 GiB memory in use. Of the allocated memory 10.38 GiB is allocated by PyTorch, and 3.70 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [14]:
print(model_uk_v2)

XLMRobertaForTokenClassification(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=5, bias=True)
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
 

In [15]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN_UKR")
login(token=hf_token)

model_uk_v2.push_to_hub("marianaY/ner-loc-date-anonymizer-uk")
tokenizer.push_to_hub("marianaY/ner-loc-date-anonymizer-uk")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/marianaY/ner-loc-date-anonymizer-uk/commit/2ccb37b941024095a178dd91afa531c892cb0614', commit_message='Upload tokenizer', commit_description='', oid='2ccb37b941024095a178dd91afa531c892cb0614', pr_url=None, repo_url=RepoUrl('https://huggingface.co/marianaY/ner-loc-date-anonymizer-uk', endpoint='https://huggingface.co', repo_type='model', repo_id='marianaY/ner-loc-date-anonymizer-uk'), pr_revision=None, pr_num=None)